In [2]:
import json
import math

In [3]:
def load_json(file_path):
    """Load JSON data from a file."""
    with open(file_path, "r") as f:
        return json.load(f)

In [4]:
def recompute_tasks_runtime(stage_id, num_tasks, avg_runtime_ms, max_executors, min_task_runtime_ms):
    """Recompute the number of tasks and runtime per task while enforcing constraints."""
    profiled_runtime = math.ceil(avg_runtime_ms / 1000)  # Convert ms to seconds

    if num_tasks > max_executors:
        adjusted_runtime = math.ceil((num_tasks * profiled_runtime) / max_executors)
        adjusted_num_tasks = max_executors
    else:
        adjusted_runtime = profiled_runtime
        adjusted_num_tasks = num_tasks

    final_runtime = max(math.ceil(min_task_runtime_ms / 1000), adjusted_runtime)  # Enforce min runtime

    # print(
    #     f"Stage {stage_id}: num_tasks ({num_tasks}) -> {adjusted_num_tasks}, "
    #     f"runtime_s ({profiled_runtime}) -> {final_runtime}"
    # )

    return adjusted_num_tasks, final_runtime

In [5]:
def extract_tpch_data(json_data, dataset_size, max_executors, min_task_runtime_ms):
    """Extract TPCH query data, filter relevant queries, and recompute runtime."""
    extracted_data = {}

    for query_key, stages in json_data.items():
        if dataset_size in query_key and "maxCores_" + str(max_executors) in query_key:
            query_id = query_key.split("_")[1]  # Extract query number (e.g., 'q1')
            # print(f"--------Processing query {query_id}")
            extracted_data[query_id] = [
                (stage["stage_id"], recompute_tasks_runtime(stage["stage_id"], stage["num_tasks"], int(stage["average_runtime_ms"]), max_executors, min_task_runtime_ms))
                for stage in stages
            ]

    return extracted_data

In [6]:
def compute_resource_space(data):
    """Compute the total resource space required for each query."""
    return {query_id: sum(num_tasks * runtime for _, (num_tasks, runtime) in stages) for query_id, stages in data.items()}

In [7]:
def bucketize_queries(resource_space, bucket_size):
    """Classify queries into easy, medium, and hard buckets based on resource consumption."""
    buckets = {"easy": [], "medium": [], "hard": []}
    for query_id, value in resource_space.items():
        if value < bucket_size:
            buckets["easy"].append(query_id)
        elif bucket_size <= value <= 2 * bucket_size:
            buckets["medium"].append(query_id)
        else:
            buckets["hard"].append(query_id)
    return buckets

In [8]:
def analyze_tpch_queries(json_path, bucket_size, dataset_size, max_executors, min_task_runtime_ms):
    """Main function to process TPCH queries and return categorized buckets."""
    json_data = load_json(json_path)
    extracted_data = extract_tpch_data(json_data, dataset_size, max_executors, min_task_runtime_ms)
    resource_requirements = compute_resource_space(extracted_data)
    return bucketize_queries(resource_requirements, bucket_size)

In [9]:
# Input params to create buckets
json_path = "/home/dgarg39/erdos-scheduling-simulator/profiles/workload/tpch/cloudlab/cloudlab_22query_tpch_profiles.json"
bucket_size=8000
dataset_size="100g"
max_executors=200
min_task_runtime_ms=12000

buckets = analyze_tpch_queries(json_path=json_path, bucket_size=bucket_size, dataset_size=dataset_size, max_executors=max_executors, min_task_runtime_ms=min_task_runtime_ms)

#### 100g dataset, varying the executors: 75, 100, 200

In [10]:
buckets = analyze_tpch_queries(json_path=json_path, bucket_size=5500, dataset_size="100g", max_executors=75, min_task_runtime_ms=12000)
buckets

{'easy': ['q11', 'q13', 'q14', 'q15', 'q19', 'q20', 'q22'],
 'medium': ['q1', 'q2', 'q4', 'q6', 'q10', 'q12', 'q16', 'q17', 'q18'],
 'hard': ['q3', 'q5', 'q7', 'q8', 'q9', 'q21']}

In [11]:
buckets = analyze_tpch_queries(json_path=json_path, bucket_size=8000, dataset_size="100g", max_executors=100, min_task_runtime_ms=12000)
buckets

{'easy': ['q2', 'q11', 'q13', 'q16', 'q19', 'q22'],
 'medium': ['q1', 'q4', 'q6', 'q10', 'q12', 'q14', 'q15', 'q17', 'q20'],
 'hard': ['q3', 'q5', 'q7', 'q8', 'q9', 'q18', 'q21']}

In [12]:
buckets = analyze_tpch_queries(json_path=json_path, bucket_size=10000, dataset_size="100g", max_executors=200, min_task_runtime_ms=12000)
buckets

{'easy': ['q6', 'q11', 'q13', 'q19', 'q22'],
 'medium': ['q1', 'q2', 'q4', 'q10', 'q12', 'q14', 'q15', 'q16', 'q20'],
 'hard': ['q3', 'q5', 'q7', 'q8', 'q9', 'q17', 'q18', 'q21']}

#### 250g dataset, varying the executors: 75, 100, 250

In [13]:
buckets = analyze_tpch_queries(json_path=json_path, bucket_size=11500, dataset_size="250g", max_executors=75, min_task_runtime_ms=12000)
buckets

{'easy': ['q2', 'q11', 'q13', 'q16', 'q19', 'q22'],
 'medium': ['q1', 'q6', 'q7', 'q10', 'q12', 'q14', 'q15', 'q20'],
 'hard': ['q3', 'q4', 'q5', 'q8', 'q9', 'q17', 'q18', 'q21']}

In [14]:
buckets = analyze_tpch_queries(json_path=json_path, bucket_size=15000, dataset_size="250g", max_executors=100, min_task_runtime_ms=12000)
buckets

{'easy': ['q2', 'q11', 'q13', 'q16', 'q19', 'q22'],
 'medium': ['q1', 'q6', 'q10', 'q12', 'q14', 'q15', 'q20'],
 'hard': ['q3', 'q4', 'q5', 'q7', 'q8', 'q9', 'q17', 'q18', 'q21']}

In [15]:
buckets = analyze_tpch_queries(json_path=json_path, bucket_size=15000, dataset_size="250g", max_executors=200, min_task_runtime_ms=12000)
buckets

{'easy': ['q1', 'q2', 'q6', 'q11', 'q13', 'q16', 'q22'],
 'medium': ['q4', 'q7', 'q10', 'q12', 'q14', 'q15', 'q19', 'q20'],
 'hard': ['q3', 'q5', 'q8', 'q9', 'q17', 'q18', 'q21']}